# GI PolicyGuard eval — per client or all clients

Inputs: `data/clients/{id}/` — debug outputs: `data/pipeline/`.

Set `CLIENT = "all"` or a single client id (e.g. `"ribkoff"`).

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from clients import ensure_pipeline_dirs, list_clients, load_client
from review import run_production_review
from gi_review import format_cost_summary

os.environ.setdefault("GI_PIPELINE_LOG", "0")
os.environ.setdefault("GI_VISION_ENABLED", "0")

CLIENT = "all"  # or "ribkoff" / "hallmark" / "cemaco" / "dfi"
ENABLE_VISION = False

ensure_pipeline_dirs(PROJECT_ROOT)
available = list_clients(PROJECT_ROOT)
selected = available if CLIENT in (None, "all", "*") else [CLIENT]
print("available:", available)
print("selected:", selected)


In [ ]:
def flags_of(summary: dict) -> set[str]:
    return {f["checkpoint_id"] for f in summary.get("flags") or []}


def run_report(client, report: Path) -> dict:
    t0 = time.perf_counter()
    run = run_production_review(
        report,
        client.checkpoints_path,
        specs_path=client.hand_specs,
        project_root=PROJECT_ROOT,
        enable_vision=ENABLE_VISION,
    )
    summary = format_cost_summary(run)
    summary["elapsed_seconds"] = round(time.perf_counter() - t0, 1)
    summary["report"] = report.name
    summary["client"] = client.id
    return summary


rows = []
for cid in selected:
    client = load_client(cid, PROJECT_ROOT)
    if not client.checkpoints_path.exists():
        print(f"SKIP {cid}: missing {client.checkpoints_path}")
        continue
    print("=" * 72)
    print("CLIENT", cid)
    print("PRECISION")
    corrected = client.corrected_reports()
    holdout = client.meta.get("eval_corrected")
    if holdout:
        corrected = [p for p in corrected if p.stem in set(holdout)]
    fp = 0
    for report in corrected:
        s = run_report(client, report)
        rows.append(s)
        fp += int(s.get("blocking_flags_count") or 0)
        print(
            f"  {report.stem}: blocking={s.get('blocking_flags_count')} "
            f"total={s.get('total_flags_count')} unable={s.get('unable_to_check_count')} "
            f"{s['elapsed_seconds']}s flags={sorted(flags_of(s))}"
        )
    print(f"  -> {fp} blocking FPs")

    print("RECALL")
    macro_p, macro_r = [], []
    for report, key in client.flawed_reports():
        expected = client.label_ids(key)
        if not expected:
            print(f"  SKIP {report.stem}: no labels for {key}")
            continue
        s = run_report(client, report)
        rows.append({**s, "label_key": key})
        caught = flags_of(s) & expected
        p = len(caught) / max(len(flags_of(s)), 1)
        r = len(caught) / max(len(expected), 1)
        macro_p.append(p)
        macro_r.append(r)
        print(
            f"  {report.stem}: P={p:.2f} R={r:.2f} "
            f"caught={sorted(caught)} missed={sorted(expected - flags_of(s))} "
            f"extra={sorted(flags_of(s) - expected)}"
        )
    if macro_p:
        print(f"  -> MACRO P={sum(macro_p)/len(macro_p):.3f} R={sum(macro_r)/len(macro_r):.3f}")

out = PROJECT_ROOT / "data/pipeline/eval/notebook_eval_results.json"
out.write_text(json.dumps(rows, indent=2) + "\n", encoding="utf-8")
print("Wrote", out)
